In [ ]:
import pandas as pd
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.metrics import confusion_matrix, classification_report, recall_score, precision_score, roc_curve, roc_auc_score, auc
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Recall, Precision, F1Score, AUC
from tensorflow.keras.regularizers import l1, l2, l1_l2

In [ ]:
df = pd.read_csv('covid.csv')

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
cols_to_drop = df.filter(regex = 'Date').columns.to_list()
s = df.isna().sum().sort_values(ascending=False)
cols_to_drop = list(set(cols_to_drop + list(s[s>0].index)))

In [ ]:
cols_to_drop

In [ ]:
cols_to_drop.append('Patient_ID')
cols_to_drop.append('Region')

In [ ]:
df.drop(
    columns = cols_to_drop,
    inplace = True
)

In [ ]:
df.info()

In [ ]:
obj_cols = df.select_dtypes(include='object').columns

In [ ]:
df[obj_cols] = df[obj_cols].astype('string')

In [ ]:
df.info()

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
oe = OrdinalEncoder()
final = pd.DataFrame(
    data = scaler.fit_transform(oe.fit_transform(df)),
    columns = df.columns
)

In [ ]:
final.head()

In [ ]:
target = final['Recovered']
final = final.drop(columns='Recovered')

In [ ]:
final.info()

In [ ]:
x_tr,x_te, y_tr, y_te = train_test_split(
    final, 
    target,
    shuffle = True,
    random_state = 42,
    test_size=0.3
)

In [ ]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'solver': ['saga', 'sag', 'lbfgs']
}

In [ ]:
scoring = {
    'accuracy': 'accuracy',
    'f1': 'f1',
    'recall': 'recall',
    'precision': 'precision'
}

In [ ]:
search = GridSearchCV(
    estimator = LogisticRegression(max_iter = 1000),
    cv = 5,
    param_grid = param_grid,
    refit = 'accuracy',
    scoring = scoring,
    n_jobs=-1
)

In [ ]:
search.fit(x_tr, y_tr)

In [ ]:
model = search.best_estimator_

In [ ]:
model.fit(x_tr, y_tr)

In [ ]:
y_pred = model.predict(x_te)

In [ ]:
y_te.value_counts()

In [ ]:
cm = confusion_matrix(y_te, y_pred)

In [ ]:
cm

In [ ]:
print(classification_report(y_te, y_pred))

In [ ]:
explainer = shap.LinearExplainer(model,x_te)
shap_values = explainer(x_te)

In [ ]:
shap.summary_plot(shap_values, x_te)

In [ ]:
shap.summary_plot(shap_values, x_te, plot_type='bar')

In [ ]:
y_proba = model.predict_proba(x_te)[:, 1]
fpr, tpr,threshold = roc_curve(y_te, y_proba)
roc_auc = auc(fpr, tpr)

f1_roc = []
for th in threshold:
    y_predict = (y_pred >= th).astype('int')
    tp = np.sum((y_predict == 1) & (y_te == 1))
    tn = np.sum((y_predict == 0) & (y_te == 0))
    fp = np.sum((y_predict == 1) & (y_te == 0))
    fn = np.sum((y_predict == 0) & (y_te == 1))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1score = 2*precision*recall/(precision + recall) if (precision + recall) > 0 else 0
    f1_roc.append(f1score)


best_idx = np.argmax(f1_roc)
best_fpr = fpr[best_idx]
best_tpr = tpr[best_idx]

In [ ]:
plt.figure(figsize=(15,8))
plt.plot(fpr, tpr, lw=2, label = f'Roc curve auc: {roc_auc:.2f}')
plt.plot([0,1], [0,1],color='lightgray', linestyle='--',label='Random Classifier')
plt.axhline(y=best_tpr,lw=2, color='green', label='True Positive Rate', linestyle='--')
plt.axvline(x=best_fpr, lw=2, color='red', label='False Positive Rate', linestyle='--')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.legend()
plt.grid(True,alpha=0.4)

In [ ]:
ml = Sequential(
    [
        Input(shape=(14,)),
        Dense(units = 128, activation='relu', kernel_regularizer = l1(0.01)),
        Dropout(0.3),
        Dense(units = 64, activation='relu', kernel_regularizer = l1(0.0001)),
        Dropout(0.2),
        Dense(units = 32, activation='relu', kernel_regularizer = l1(0.001)),
        Dropout(0.4),
        Dense(units = 16, activation='relu', kernel_regularizer = l1(0.0001)),
        Dropout(0.3),
        Dense(units = 1, activation='softmax')
    ]
)

In [ ]:
ml.compile(
    metrics = [
        'accuracy',
        Recall(name='recall'),
        Precision(name='precision'),
        F1Score(name='f1-score'), 
        AUC(name='auc')
    ],
    loss = 'binary_crossentropy',
    optimizer = Adam(learning_rate=0.001)
)

In [ ]:
ml.summary()

In [ ]:
history = ml.fit(
    x_tr,
    y_tr,
    epochs=20,
    batch_size=15,
    verbose=2,
    validation_split=0.3
)

In [ ]:
y_pred = ml.predict(x_te)

In [ ]:
print(classification_report(y_te, y_pred))

In [ ]:
plt.figure(figsize=(15, 8))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label = 'Train Accuracy')
plt.plot(history.history['val_accuracy'], label = 'Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label = 'Train Loss')
plt.plot(history.history['val_loss'], label = 'Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()